# Individual Task 1 — Part 1.3: Data Analysis

COSC2669/COSC2816 Case Studies in Data Science

Two public healthcare datasets, two classification algorithms each.

| | |
|---|---|
| Dataset A | Diabetes 130-US Hospitals, 1999–2008 (UCI id 296) |
| Dataset B | Heart Disease, Cleveland (UCI id 45) |
| Model 1 | Logistic regression, L2 penalty, balanced class weights |
| Model 2 | Histogram-based gradient boosting |

Neither model was used in Practical Data Science, which used decision trees,
k-nearest neighbours, random forests and item-based collaborative filtering.

Run the cells in order. The final cell prints every number needed for the
report, formatted for copy-paste.

In [1]:
# --- Setup -------------------------------------------------------------
# Installs on first run; harmless afterwards.
import importlib.util
import subprocess
import sys

for pkg in ["ucimlrepo", "scikit-learn", "pandas", "matplotlib"]:
    mod = "sklearn" if pkg == "scikit-learn" else pkg
    if importlib.util.find_spec(mod) is None:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
os.makedirs("data", exist_ok=True)
os.makedirs("figures", exist_ok=True)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)

SKLEARN_VERSION = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
print(f"scikit-learn {sklearn.__version__}  |  pandas {pd.__version__}")
if SKLEARN_VERSION < (1, 4):
    print("NOTE: scikit-learn below 1.4 — gradient boosting will use manual "
          "class weighting instead of the class_weight argument.")

scikit-learn 1.7.2  |  pandas 2.3.3


## 1. Loading

`ucimlrepo` splits a dataset into `features`, `targets` and `ids`. The
diabetes data declares `encounter_id` and `patient_nbr` as ID-role variables,
so they land in `ids`, not `features`. Joining only features and targets drops
them, which silently disables the deduplication in section 2 and leaves the
same patients in both the training and test partitions.

`repo.data.original` is the complete frame before any splitting, so it is used
in preference. The cache is validated before use, since a file written by an
earlier version would otherwise reproduce the problem.

In [2]:
REQUIRED_DIABETES_COLS = [
    "patient_nbr", "discharge_disposition_id", "readmitted", "weight",
]


def _cache_is_valid(path, required):
    """A cache missing required columns is worse than no cache."""
    try:
        head = pd.read_csv(path, nrows=1)
    except Exception:
        return False
    return all(c in head.columns for c in required)


def load_uci(dataset_id, cache_name, required=()):
    """Fetch a UCI dataset, caching to CSV so later runs work offline."""
    cache = os.path.join("data", f"{cache_name}.csv")

    if os.path.exists(cache):
        if _cache_is_valid(cache, required):
            df = pd.read_csv(cache, low_memory=False)
            print(f"Loaded cached {cache_name}: {df.shape}")
            return df
        os.remove(cache)
        print(f"Discarded incomplete cache: {cache}")

    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=dataset_id)

    # Preferred: the untouched original frame.
    df = getattr(repo.data, "original", None)

    # Fallback: reassemble from the separate blocks.
    if df is None or not len(getattr(df, "columns", [])):
        blocks = []
        for attr in ("ids", "features", "targets"):
            block = getattr(repo.data, attr, None)
            if block is not None and len(getattr(block, "columns", [])):
                blocks.append(block)
        if not blocks:
            raise RuntimeError(f"ucimlrepo returned no usable data for id={dataset_id}")
        df = pd.concat(blocks, axis=1)

    df = df.loc[:, ~df.columns.duplicated()].copy()
    df.to_csv(cache, index=False)
    print(f"Downloaded {cache_name}: {df.shape}")
    return df


diabetes_raw = load_uci(296, "diabetes_130us", REQUIRED_DIABETES_COLS)
heart_raw = load_uci(45, "heart_cleveland")

print(f"\nDiabetes: {diabetes_raw.shape[0]:,} rows x {diabetes_raw.shape[1]} columns")
print(f"Heart:    {heart_raw.shape[0]:,} rows x {heart_raw.shape[1]} columns")

missing = [c for c in REQUIRED_DIABETES_COLS if c not in diabetes_raw.columns]
if missing:
    print(f"\nMissing: {missing}")
    print(f"Received {len(diabetes_raw.columns)} columns:")
    print(sorted(diabetes_raw.columns))
    raise RuntimeError(
        "Identifier columns absent. Manual route: download "
        "https://archive.ics.uci.edu/static/public/296/"
        "diabetes+130+us+hospitals+for+years+1999+2008.zip, extract "
        "diabetic_data.csv, and save it as data/diabetes_130us.csv."
    )

print(f"\nCheck passed: {diabetes_raw['patient_nbr'].nunique():,} unique patients "
      f"across {len(diabetes_raw):,} encounters.")

Downloaded diabetes_130us: (101766, 50)
Downloaded heart_cleveland: (303, 14)

Diabetes: 101,766 rows x 50 columns
Heart:    303 rows x 14 columns

Check passed: 71,518 unique patients across 101,766 encounters.


## 2. Dataset A — cleaning

Each decision below is defensible in the write-up:

| Step | Reason |
|---|---|
| Convert `?` to NaN | Missing values are string-encoded, so every column reads as object otherwise |
| Drop `weight` | ~97% missing; imputation at that rate constructs data rather than recovering it |
| Keep "Unknown" as a level for `payer_code` and `medical_specialty` | The missingness itself carries information |
| Drop discharge codes 11, 13, 14, 19, 20, 21 | Death and hospice transfers cannot be readmitted |
| Keep the first encounter per patient | Repeat encounters leak across the train/test split |
| Drop `examide`, `citoglipton` | Single-valued, zero variance |
| Group ICD-9 into chapters | 700+ raw codes would explode the one-hot dimensionality |

In [3]:
DEAD_OR_HOSPICE = [11, 13, 14, 19, 20, 21]


def group_icd9(code):
    """Collapse an ICD-9 code into a clinical chapter (Strack et al., 2014)."""
    if pd.isna(code):
        return "Missing"
    s = str(code).strip()
    if s.startswith(("V", "E")):
        return "Other"
    try:
        v = float(s)
    except ValueError:
        return "Other"
    iv = int(v)
    if 390 <= v < 460 or iv == 785:
        return "Circulatory"
    if 460 <= v < 520 or iv == 786:
        return "Respiratory"
    if 520 <= v < 580 or iv == 787:
        return "Digestive"
    if iv == 250:
        return "Diabetes"
    if 800 <= v < 1000:
        return "Injury"
    if 710 <= v < 740:
        return "Musculoskeletal"
    if 580 <= v < 630 or iv == 788:
        return "Genitourinary"
    if 140 <= v < 240:
        return "Neoplasms"
    return "Other"


def prepare_diabetes(df):
    df = df.replace("?", np.nan).copy()
    n_start = len(df)

    if "discharge_disposition_id" in df.columns:
        codes = pd.to_numeric(df["discharge_disposition_id"], errors="coerce")
        df = df[~codes.isin(DEAD_OR_HOSPICE)]
    n_after_deaths = len(df)

    if "patient_nbr" in df.columns:
        df = df.drop_duplicates(subset="patient_nbr", keep="first")
    n_after_dedup = len(df)

    df["target"] = (df["readmitted"] == "<30").astype(int)

    for c in ["diag_1", "diag_2", "diag_3"]:
        if c in df.columns:
            df[c + "_grp"] = df[c].apply(group_icd9)

    df["prior_visits"] = sum(
        pd.to_numeric(df[c], errors="coerce").fillna(0)
        for c in ["number_outpatient", "number_emergency", "number_inpatient"]
        if c in df.columns
    )

    drop = ["encounter_id", "patient_nbr", "weight", "readmitted",
            "examide", "citoglipton", "diag_1", "diag_2", "diag_3"]
    df = df.drop(columns=[c for c in drop if c in df.columns])

    for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
        if c in df.columns:
            df[c] = df[c].astype(str)

    print(f"  {n_start:,} encounters")
    print(f"  -{n_start - n_after_deaths:,} death and hospice discharges")
    print(f"  -{n_after_deaths - n_after_dedup:,} repeat encounters (one kept per patient)")
    print(f"  = {n_after_dedup:,} retained")
    return df.reset_index(drop=True)


print("Cleaning Dataset A:")
dia = prepare_diabetes(diabetes_raw)
y_dia = dia.pop("target")
X_dia = dia

assert len(X_dia) < 0.85 * len(diabetes_raw), (
    "Too few rows removed — deduplication did not run. Check that patient_nbr "
    "survived loading."
)

N_DIA, K_DIA = X_dia.shape
P_DIA = y_dia.mean()

print(f"\nFinal: {N_DIA:,} encounters, {K_DIA} features")
print(f"Positive class (readmitted within 30 days): {P_DIA:.1%}")
print(f"Predicting 'no readmission' for everyone scores {1 - P_DIA:.1%} accuracy "
      "while identifying nobody.")

Cleaning Dataset A:
  101,766 encounters
  -2,423 death and hospice discharges
  -29,353 repeat encounters (one kept per patient)
  = 69,990 retained

Final: 69,990 encounters, 45 features
Positive class (readmitted within 30 days): 9.0%
Predicting 'no readmission' for everyone scores 91.0% accuracy while identifying nobody.


## 3. Dataset A — preprocessing pipeline

`age` arrives as brackets, so it is ordinal rather than nominal. Numeric
columns are scaled: required by logistic regression, harmless for the trees.

In [4]:
AGE_ORDER = [["[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
              "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)"]]

ohe_kwargs = {"handle_unknown": "ignore"}
if SKLEARN_VERSION >= (1, 2):
    ohe_kwargs["sparse_output"] = False
else:
    ohe_kwargs["sparse"] = False
if SKLEARN_VERSION >= (1, 1):
    ohe_kwargs["min_frequency"] = 30

num_cols = X_dia.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X_dia.columns if c not in num_cols and c != "age"]
has_age = "age" in X_dia.columns

transformers = [
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("oh", OneHotEncoder(**ohe_kwargs))]), cat_cols),
]
if has_age:
    transformers.insert(1, (
        "age",
        Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                  ("ord", OrdinalEncoder(categories=AGE_ORDER,
                                         handle_unknown="use_encoded_value",
                                         unknown_value=-1)),
                  ("sc", StandardScaler())]),
        ["age"],
    ))

pre_dia = ColumnTransformer(transformers, remainder="drop")

X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_dia, y_dia, test_size=0.25, stratify=y_dia, random_state=RANDOM_STATE)
print(f"Train {len(X_tr_d):,} / Test {len(X_te_d):,}")
print(f"{len(num_cols)} numeric, {len(cat_cols)} categorical"
      f"{', 1 ordinal (age)' if has_age else ''}")

Train 52,492 / Test 17,498
9 numeric, 35 categorical, 1 ordinal (age)


## 4. Dataset A — models

Both are class-weighted so neither simply learns the majority class.

In [5]:
hgb_kwargs = dict(max_iter=300, learning_rate=0.06, max_leaf_nodes=31,
                  l2_regularization=1.0, early_stopping=True,
                  validation_fraction=0.15, random_state=RANDOM_STATE)
if SKLEARN_VERSION >= (1, 4):
    hgb_kwargs["class_weight"] = "balanced"

logreg_dia = Pipeline([
    ("pre", pre_dia),
    ("clf", LogisticRegression(max_iter=2000, C=0.1, penalty="l2",
                               class_weight="balanced", solver="lbfgs",
                               random_state=RANDOM_STATE)),
])
hgb_dia = Pipeline([
    ("pre", pre_dia),
    ("clf", HistGradientBoostingClassifier(**hgb_kwargs)),
])

models_dia = {"Logistic Regression": logreg_dia, "Gradient Boosting": hgb_dia}

for name, model in models_dia.items():
    print(f"Fitting {name} ...")
    if name == "Gradient Boosting" and SKLEARN_VERSION < (1, 4):
        w = np.where(y_tr_d == 1, (y_tr_d == 0).sum() / (y_tr_d == 1).sum(), 1.0)
        model.fit(X_tr_d, y_tr_d, clf__sample_weight=w)
    else:
        model.fit(X_tr_d, y_tr_d)
print("Done.")

Fitting Logistic Regression ...
Fitting Gradient Boosting ...
Done.


## 5. Evaluation

`precision_at_k` is the operationally honest metric here. A care-coordination
team can only contact a fixed number of patients per month, so what matters is
the hit rate inside the top decile of predicted risk, not global accuracy.

In [6]:
def precision_at_k(y_true, scores, k_frac=0.10):
    y_true = np.asarray(y_true)
    n = max(1, int(len(scores) * k_frac))
    top = np.argsort(scores)[::-1][:n]
    return y_true[top].mean()


def recall_at_k(y_true, scores, k_frac=0.10):
    y_true = np.asarray(y_true)
    n = max(1, int(len(scores) * k_frac))
    top = np.argsort(scores)[::-1][:n]
    return y_true[top].sum() / max(1, y_true.sum())


def evaluate(name, model, X_test, y_test, threshold=0.5, include_at_k=True):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Balanced acc": balanced_accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Brier": brier_score_loss(y_test, proba),
    }
    if include_at_k:
        row["Prec@10%"] = precision_at_k(y_test, proba)
        row["Recall@10%"] = recall_at_k(y_test, proba)
    return row, proba, pred


rows_d, proba_d = [], {}
for name, model in models_dia.items():
    row, proba, pred = evaluate(name, model, X_te_d, y_te_d)
    rows_d.append(row)
    proba_d[name] = proba
    print(f"\n--- {name} ---")
    print(classification_report(y_te_d, pred,
                                target_names=["Not readmitted", "Readmitted <30d"],
                                digits=3, zero_division=0))
    print(pd.DataFrame(confusion_matrix(y_te_d, pred),
                       index=["Actual 0", "Actual 1"],
                       columns=["Pred 0", "Pred 1"]))

results_dia = pd.DataFrame(rows_d).set_index("Model").round(3)
print("\n=== Dataset A: 30-day readmission ===")
print(results_dia.T)


--- Logistic Regression ---
                 precision    recall  f1-score   support

 Not readmitted      0.938     0.675     0.785     15927
Readmitted <30d      0.142     0.544     0.225      1571

       accuracy                          0.663     17498
      macro avg      0.540     0.609     0.505     17498
   weighted avg      0.866     0.663     0.734     17498

          Pred 0  Pred 1
Actual 0   10745    5182
Actual 1     716     855

--- Gradient Boosting ---
                 precision    recall  f1-score   support

 Not readmitted      0.938     0.663     0.777     15927
Readmitted <30d      0.140     0.558     0.224      1571

       accuracy                          0.654     17498
      macro avg      0.539     0.610     0.501     17498
   weighted avg      0.867     0.654     0.727     17498

          Pred 0  Pred 1
Actual 0   10563    5364
Actual 1     695     876

=== Dataset A: 30-day readmission ===
Model         Logistic Regression  Gradient Boosting
Accuracy    

## 6. Dataset B — Heart Disease (Cleveland)

303 records. A single split is unstable at that size, so the headline figures
come from stratified 10-fold cross-validation with standard deviations.

In [7]:
def prepare_heart(df):
    df = df.replace("?", np.nan).copy()
    target_col = "num" if "num" in df.columns else df.columns[-1]
    df["target"] = (pd.to_numeric(df[target_col], errors="coerce") > 0).astype(int)
    df = df.drop(columns=[target_col])
    for c in df.columns:
        if c != "target":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.reset_index(drop=True)


hrt = prepare_heart(heart_raw)
y_hrt = hrt.pop("target")
X_hrt = hrt
P_HRT = y_hrt.mean()

print(f"{len(X_hrt)} patients, {X_hrt.shape[1]} attributes")
print(f"Positive class (disease present): {P_HRT:.1%}")
na = X_hrt.isna().sum()
print("Missing values:", dict(na[na > 0]) or "none")

NOMINAL = [c for c in ["cp", "restecg", "slope", "thal"] if c in X_hrt.columns]
NUMERIC = [c for c in X_hrt.columns if c not in NOMINAL]

ohe_h = {k: v for k, v in ohe_kwargs.items() if k != "min_frequency"}
pre_hrt = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(**ohe_h))]), NOMINAL),
])

models_hrt = {
    "Logistic Regression": Pipeline([
        ("pre", pre_hrt),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced",
                                   random_state=RANDOM_STATE))]),
    "Gradient Boosting": Pipeline([
        ("pre", pre_hrt),
        ("clf", HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.05, max_leaf_nodes=8,
            min_samples_leaf=10, l2_regularization=1.0,
            random_state=RANDOM_STATE))]),
}

303 patients, 13 attributes
Positive class (disease present): 45.9%
Missing values: {'ca': np.int64(4), 'thal': np.int64(2)}


In [8]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "balanced_accuracy", "recall", "precision", "f1",
           "roc_auc", "average_precision"]

cv_rows, cv_raw = [], {}
for name, model in models_hrt.items():
    res = cross_validate(model, X_hrt, y_hrt, cv=cv, scoring=scoring, n_jobs=-1)
    cv_raw[name] = res
    cv_rows.append({"Model": name, **{
        s: f"{res['test_' + s].mean():.3f} ± {res['test_' + s].std():.3f}"
        for s in scoring}})

results_hrt = pd.DataFrame(cv_rows).set_index("Model")
print("=== Dataset B: stratified 10-fold CV ===")
print(results_hrt.T)

X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(
    X_hrt, y_hrt, test_size=0.25, stratify=y_hrt, random_state=RANDOM_STATE)
proba_h = {}
for name, model in models_hrt.items():
    model.fit(X_tr_h, y_tr_h)
    proba_h[name] = model.predict_proba(X_te_h)[:, 1]
    print(f"\n--- {name} (hold-out) ---")
    print(classification_report(y_te_h, model.predict(X_te_h),
                                target_names=["No disease", "Disease"],
                                digits=3, zero_division=0))

=== Dataset B: stratified 10-fold CV ===
Model             Logistic Regression Gradient Boosting
accuracy                0.841 ± 0.049     0.819 ± 0.051
balanced_accuracy       0.839 ± 0.051     0.816 ± 0.048
recall                  0.799 ± 0.083     0.776 ± 0.070
precision               0.850 ± 0.064     0.835 ± 0.109
f1                      0.821 ± 0.059     0.798 ± 0.052
roc_auc                 0.913 ± 0.043     0.889 ± 0.049
average_precision       0.909 ± 0.050     0.883 ± 0.062

--- Logistic Regression (hold-out) ---
              precision    recall  f1-score   support

  No disease      0.875     0.854     0.864        41
     Disease      0.833     0.857     0.845        35

    accuracy                          0.855        76
   macro avg      0.854     0.855     0.855        76
weighted avg      0.856     0.855     0.855        76


--- Gradient Boosting (hold-out) ---
              precision    recall  f1-score   support

  No disease      0.872     0.829     0.850        

## 7. Figures

The precision-recall panel is the informative one for the imbalanced diabetes
data; ROC flatters imbalanced classifiers.

In [9]:
def plot_curves(y_true, probas, title, fname):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

    for name, p in probas.items():
        fpr, tpr, _ = roc_curve(y_true, p)
        ax[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_score(y_true, p):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=1)
    ax[0].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC")
    ax[0].legend(fontsize=8, loc="lower right")

    for name, p in probas.items():
        pr, rc, _ = precision_recall_curve(y_true, p)
        ax[1].plot(rc, pr, lw=2,
                   label=f"{name} (AP={average_precision_score(y_true, p):.3f})")
    ax[1].axhline(np.mean(y_true), color="k", ls="--", lw=1, label="Prevalence")
    ax[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
    ax[1].legend(fontsize=8)

    for name, p in probas.items():
        frac, mean_pred = calibration_curve(y_true, p, n_bins=10, strategy="quantile")
        ax[2].plot(mean_pred, frac, "o-", lw=2, label=name)
    ax[2].plot([0, 1], [0, 1], "k--", lw=1)
    ax[2].set(xlabel="Mean predicted probability", ylabel="Observed frequency",
              title="Calibration")
    ax[2].legend(fontsize=8)

    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    fig.savefig(f"figures/{fname}", dpi=160, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figures/{fname}")


plot_curves(y_te_d, proba_d, "Dataset A — 30-day readmission", "diabetes_curves.png")
plot_curves(y_te_h, proba_h, "Dataset B — heart disease presence", "heart_curves.png")

Saved figures/diabetes_curves.png
Saved figures/heart_curves.png


## 8. What drives the predictions

Signed logistic coefficients give direction and magnitude; permutation
importance is model-agnostic and captures non-linear contributions.

In [10]:
def top_coefficients(pipe, n=12):
    names = pipe.named_steps["pre"].get_feature_names_out()
    coefs = pipe.named_steps["clf"].coef_[0]
    s = pd.Series(coefs, index=names)
    return pd.concat([s.nlargest(n), s.nsmallest(n)]).sort_values(ascending=False)


print("=== Dataset A: logistic coefficients (log-odds) ===")
print(top_coefficients(models_dia["Logistic Regression"]).round(3))

print("\n=== Dataset B: logistic coefficients (log-odds) ===")
print(top_coefficients(models_hrt["Logistic Regression"], n=8).round(3))

=== Dataset A: logistic coefficients (log-odds) ===
cat__discharge_disposition_id_22                            0.971
cat__medical_specialty_Hematology/Oncology                  0.920
cat__discharge_disposition_id_15                            0.882
cat__discharge_disposition_id_28                            0.831
cat__discharge_disposition_id_5                             0.655
cat__medical_specialty_Oncology                             0.565
cat__admission_source_id_3                                  0.470
cat__repaglinide_Up                                         0.424
cat__medical_specialty_Psychiatry                           0.410
cat__medical_specialty_Nephrology                           0.388
cat__diag_2_grp_Neoplasms                                   0.371
cat__medical_specialty_PhysicalMedicineandRehabilitation    0.361
cat__discharge_disposition_id_6                            -0.315
cat__admission_source_id_5                                 -0.321
cat__tolazamide_infreque

In [11]:
sub = min(6000, len(X_te_d))
perm_d = permutation_importance(
    models_dia["Gradient Boosting"], X_te_d.iloc[:sub], y_te_d.iloc[:sub],
    scoring="average_precision", n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=-1)
imp_d = pd.Series(perm_d.importances_mean, index=X_te_d.columns).sort_values(ascending=False)
print("=== Dataset A: permutation importance (drop in PR-AUC) ===")
print(imp_d.head(12).round(5))

perm_h = permutation_importance(
    models_hrt["Gradient Boosting"], X_te_h, y_te_h,
    scoring="roc_auc", n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
imp_h = pd.Series(perm_h.importances_mean, index=X_te_h.columns).sort_values(ascending=False)
print("\n=== Dataset B: permutation importance (drop in ROC-AUC) ===")
print(imp_h.head(10).round(4))

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
imp_d.head(10).iloc[::-1].plot.barh(ax=ax[0], color="#4c72b0")
ax[0].set(title="Dataset A — permutation importance", xlabel="Drop in PR-AUC")
imp_h.head(10).iloc[::-1].plot.barh(ax=ax[1], color="#dd8452")
ax[1].set(title="Dataset B — permutation importance", xlabel="Drop in ROC-AUC")
fig.tight_layout()
fig.savefig("figures/feature_importance.png", dpi=160, bbox_inches="tight")
plt.close(fig)
print("\nSaved figures/feature_importance.png")

=== Dataset A: permutation importance (drop in PR-AUC) ===
discharge_disposition_id    0.05101
number_inpatient            0.01999
medical_specialty           0.00571
diag_2_grp                  0.00420
prior_visits                0.00369
payer_code                  0.00338
number_diagnoses            0.00257
time_in_hospital            0.00254
diag_1_grp                  0.00239
admission_source_id         0.00200
number_emergency            0.00200
num_lab_procedures          0.00196
dtype: float64

=== Dataset B: permutation importance (drop in ROC-AUC) ===
cp         0.0504
ca         0.0494
thal       0.0200
sex        0.0192
oldpeak    0.0192
slope      0.0084
age        0.0073
exang      0.0073
chol       0.0067
thalach    0.0049
dtype: float64

Saved figures/feature_importance.png


## 9. Threshold selection

The 0.5 cut-off is arbitrary. A missed readmission costs far more than an
unnecessary follow-up call, so the table below supports stating a defensible
operating point rather than accepting the default.

In [12]:
def threshold_table(y_true, proba, thresholds=(0.30, 0.40, 0.50, 0.60, 0.70)):
    out = []
    for t in thresholds:
        pred = (proba >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        out.append({
            "Threshold": t, "Flagged": int(pred.sum()),
            "TP": tp, "FP": fp, "FN": fn,
            "Precision": round(precision_score(y_true, pred, zero_division=0), 3),
            "Recall": round(recall_score(y_true, pred, zero_division=0), 3),
            "Specificity": round(tn / (tn + fp), 3) if (tn + fp) else np.nan,
        })
    return pd.DataFrame(out)


print("=== Dataset A: gradient boosting, threshold sensitivity ===")
print(threshold_table(y_te_d, proba_d["Gradient Boosting"]).to_string(index=False))

=== Dataset A: gradient boosting, threshold sensitivity ===
 Threshold  Flagged   TP    FP   FN  Precision  Recall  Specificity
       0.3    15710 1517 14193   54      0.097   0.966        0.109
       0.4    11613 1285 10328  286      0.111   0.818        0.352
       0.5     6240  876  5364  695      0.140   0.558        0.663
       0.6     1883  405  1478 1166      0.215   0.258        0.907
       0.7      577  170   407 1401      0.295   0.108        0.974


## 10. Report values

Everything needed for the write-up, in one place.

In [13]:
gb_d = results_dia.loc["Gradient Boosting"]
lr_d = results_dia.loc["Logistic Regression"]
hrt_auc = {n: cv_raw[n]["test_roc_auc"].mean() for n in models_hrt}
best_h = max(hrt_auc, key=hrt_auc.get)

print("=" * 68)
print("VALUES FOR 03-analysis.tex")
print("=" * 68)
print(f"  [N]  encounters after cleaning ... {N_DIA:,}")
print(f"  [K]  features after cleaning ..... {K_DIA}")
print(f"  [P]  positive class .............. {P_DIA:.1%}")
print(f"  [100-P] majority accuracy ........ {1 - P_DIA:.1%}")
print(f"  [P2] Cleveland positive class .... {P_HRT:.1%}")
print(f"  [A]  Cleveland ROC-AUC ({best_h[:4]}) ... {hrt_auc[best_h]:.3f}")
print(f"  [B]  diabetes ROC-AUC (GB) ....... {gb_d['ROC-AUC']:.3f}")
print(f"\n  Prec@10% (GB) .................... {gb_d['Prec@10%']:.3f}")
print(f"  Lift over base rate .............. {gb_d['Prec@10%'] / P_DIA:.1f}x")
print(f"  Recall@10% (GB) .................. {gb_d['Recall@10%']:.3f}")
print(f"  Brier: LR {lr_d['Brier']:.3f} vs GB {gb_d['Brier']:.3f}")

print(f"\n  Top diabetes features: {', '.join(imp_d.head(3).index)}")
print(f"  Top Cleveland features: {', '.join(imp_h.head(3).index)}")

print("\n" + "=" * 68)
print("TABLE 2 ROWS FOR 99-appendix.tex")
print("=" * 68)
for n in models_dia:
    r = results_dia.loc[n]
    print(f"    Diabetes, {n.lower():<20} & {r['Recall']:.3f} & {r['ROC-AUC']:.3f} "
          f"& {r['PR-AUC']:.3f} & {r['Prec@10%']:.3f} \\\\")
for n in models_hrt:
    res = cv_raw[n]
    print(f"    Cleveland, {n.lower():<19} & {res['test_recall'].mean():.3f} "
          f"& {res['test_roc_auc'].mean():.3f} "
          f"& {res['test_average_precision'].mean():.3f} & --- \\\\")

results_dia.to_csv("figures/results_diabetes.csv")
results_hrt.to_csv("figures/results_heart.csv")
print("\nSaved results CSVs and 3 figures to ./figures/")

VALUES FOR 03-analysis.tex
  [N]  encounters after cleaning ... 69,990
  [K]  features after cleaning ..... 45
  [P]  positive class .............. 9.0%
  [100-P] majority accuracy ........ 91.0%
  [P2] Cleveland positive class .... 45.9%
  [A]  Cleveland ROC-AUC (Logi) ... 0.913
  [B]  diabetes ROC-AUC (GB) ....... 0.657

  Prec@10% (GB) .................... 0.222
  Lift over base rate .............. 2.5x
  Recall@10% (GB) .................. 0.248
  Brier: LR 0.227 vs GB 0.218

  Top diabetes features: discharge_disposition_id, number_inpatient, medical_specialty
  Top Cleveland features: cp, ca, thal

TABLE 2 ROWS FOR 99-appendix.tex
    Diabetes, logistic regression  & 0.544 & 0.654 & 0.169 & 0.211 \\
    Diabetes, gradient boosting    & 0.558 & 0.657 & 0.181 & 0.222 \\
    Cleveland, logistic regression & 0.799 & 0.913 & 0.909 & --- \\
    Cleveland, gradient boosting   & 0.776 & 0.889 & 0.883 & --- \\

Saved results CSVs and 3 figures to ./figures/
